<a href="https://colab.research.google.com/github/GordxRohit/Breast-Cancer-Detection-using-SVM-/blob/main/Svm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# 1. Load Dataset
df = pd.read_csv("Jharkhand_Breast_Cancer_Dataset.csv")

# 2. Preprocessing
df['Diagnosis_Encoded'] = df['Diagnosis'].map({
    'Benign': 0,
    'Malignant': 1
})

X = df[['Age', 'Tumor_Size_mm', 'Lymph_Node_Status',
        'Family_History', 'BMI']]

y = df['Diagnosis_Encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train SVM Model
svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale'
)

svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)

# 4. Train Decision Tree Model
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

# 5. Calculate Performance Metrics
svm_acc = accuracy_score(y_test, svm_pred)
dt_acc = accuracy_score(y_test, dt_pred)

svm_precision = precision_score(y_test, svm_pred, pos_label=1)
svm_recall = recall_score(y_test, svm_pred, pos_label=1)
svm_f1 = f1_score(y_test, svm_pred, pos_label=1)

dt_precision = precision_score(y_test, dt_pred, pos_label=1)
dt_recall = recall_score(y_test, dt_pred, pos_label=1)
dt_f1 = f1_score(y_test, dt_pred, pos_label=1)

print("SVM Accuracy:", round(svm_acc * 100, 2), "%")
print("Decision Tree Accuracy:", round(dt_acc * 100, 2), "%")

print("\nSVM Classification Report:")
print(classification_report(
    y_test, svm_pred,
    target_names=['Benign', 'Malignant']
))

print("\nSVM Confusion Matrix:")
print(confusion_matrix(y_test, svm_pred, labels=[0, 1]))

# 6. Figure 1: Dataset Distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

fig.suptitle(
    'Dataset Distribution Analysis',
    fontsize=16,
    fontweight='bold'
)

axes[0, 0].hist(
    df[df['Diagnosis'] == 'Benign']['Age'],
    bins=15, alpha=0.7, label='Benign', color='green'
)

axes[0, 0].hist(
    df[df['Diagnosis'] == 'Malignant']['Age'],
    bins=15, alpha=0.7, label='Malignant', color='red'
)

axes[0, 0].set_xlabel('Age (years)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Age Distribution by Diagnosis')
axes[0, 0].legend()

axes[0, 1].hist(
    df[df['Diagnosis'] == 'Benign']['Tumor_Size_mm'],
    bins=15, alpha=0.7, label='Benign', color='green'
)

axes[0, 1].hist(
    df[df['Diagnosis'] == 'Malignant']['Tumor_Size_mm'],
    bins=15, alpha=0.7, label='Malignant', color='red'
)

axes[0, 1].set_xlabel('Tumor Size (mm)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Tumor Size Distribution')
axes[0, 1].legend()

diagnosis_counts = df['Diagnosis'].value_counts()

axes[1, 0].pie(
    diagnosis_counts,
    labels=diagnosis_counts.index,
    autopct='%1.1f%%',
    colors=['green', 'red']
)

axes[1, 0].set_title('Diagnosis Distribution')

region_counts = df['Region'].value_counts()

axes[1, 1].bar(
    region_counts.index,
    region_counts.values,
    color='steelblue'
)

axes[1, 1].set_xlabel('Region')
axes[1, 1].set_ylabel('Record Count')
axes[1, 1].set_title('Regional Distribution')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('figure1_dataset_distribution.png',
            dpi=300, bbox_inches='tight')
plt.close()

# 7. Figure 2: SVM Confusion Matrix
cm = confusion_matrix(y_test, svm_pred, labels=[0, 1])

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Benign', 'Malignant'],
    yticklabels=['Benign', 'Malignant']
)

plt.xlabel('Predicted Diagnosis')
plt.ylabel('Actual Diagnosis')
plt.title('Confusion Matrix - SVM')

plt.tight_layout()
plt.savefig('figure2_confusion_matrix.png',
            dpi=300, bbox_inches='tight')
plt.close()

# 8. Figure 3: Model Comparison
models = ['SVM (RBF)', 'Decision Tree']

accuracies = [
    svm_acc * 100,
    dt_acc * 100
]

colors = ['#2ecc71', '#3498db']

plt.figure(figsize=(8, 6))

bars = plt.bar(
    models,
    accuracies,
    color=colors,
    edgecolor='black'
)

plt.ylabel('Accuracy (%)')
plt.title('SVM vs Decision Tree Performance')
plt.ylim(0, 100)

for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{acc:.2f}%',
        ha='center',
        fontweight='bold'
    )

plt.tight_layout()
plt.savefig('figure3_model_comparison.png',
            dpi=300, bbox_inches='tight')
plt.close()

# 9. Figure 4: Correlation Heatmap
correlation_matrix = df[
    ['Age', 'Tumor_Size_mm', 'Lymph_Node_Status',
     'Family_History', 'BMI', 'Diagnosis_Encoded']
].corr()

plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f',
    square=True
)

plt.title('Feature Correlation Heatmap')
plt.tight_layout()

plt.savefig('figure4_correlation_heatmap.png',
            dpi=300, bbox_inches='tight')
plt.close()

# 10. Figure 5: SVM Decision Boundary
X_2d = df[['Age', 'Tumor_Size_mm']].values
y_2d = df['Diagnosis_Encoded'].values

scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

svm_2d = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_2d.fit(X_2d_scaled, y_2d)

h = 0.02

x_min = X_2d_scaled[:, 0].min() - 1
x_max = X_2d_scaled[:, 0].max() + 1
y_min = X_2d_scaled[:, 1].min() - 1
y_max = X_2d_scaled[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, h),
    np.arange(y_min, y_max, h)
)

Z = svm_2d.predict(
    np.c_[xx.ravel(), yy.ravel()]
)

Z = Z.reshape(xx.shape)

plt.figure(figsize=(9, 7))

plt.contourf(
    xx, yy, Z,
    alpha=0.3,
    cmap='RdYlGn'
)

plt.scatter(
    X_2d_scaled[:, 0],
    X_2d_scaled[:, 1],
    c=y_2d,
    cmap='RdYlGn',
    edgecolors='k',
    s=60,
    alpha=0.7
)

plt.xlabel('Age (Standardized)')
plt.ylabel('Tumor Size (Standardized)')
plt.title('SVM Decision Boundary (2D)')

plt.tight_layout()

plt.savefig('figure5_svm_decision_boundary.png',
            dpi=300, bbox_inches='tight')
plt.close()

# 11. Final Results
print("\nFINAL MODEL PERFORMANCE")
print("------------------------")

print("SVM Accuracy: {:.2f}%".format(svm_acc * 100))
print("SVM Precision: {:.2f}%".format(svm_precision * 100))
print("SVM Recall: {:.2f}%".format(svm_recall * 100))
print("SVM F1-Score: {:.2f}%".format(svm_f1 * 100))

print("\nDecision Tree Accuracy: {:.2f}%".format(dt_acc * 100))
print("Decision Tree Precision: {:.2f}%".format(dt_precision * 100))
print("Decision Tree Recall: {:.2f}%".format(dt_recall * 100))
print("Decision Tree F1-Score: {:.2f}%".format(dt_f1 * 100))

print("\nAll 5 figures generated successfully!")


SVM Accuracy: 88.6 %
Decision Tree Accuracy: 84.21 %

SVM Classification Report:
              precision    recall  f1-score   support

      Benign       0.94      0.87      0.90        70
   Malignant       0.82      0.91      0.86        44

    accuracy                           0.89       114
   macro avg       0.88      0.89      0.88       114
weighted avg       0.89      0.89      0.89       114


SVM Confusion Matrix:
[[61  9]
 [ 4 40]]

FINAL MODEL PERFORMANCE
------------------------
SVM Accuracy: 88.60%
SVM Precision: 81.63%
SVM Recall: 90.91%
SVM F1-Score: 86.02%

Decision Tree Accuracy: 84.21%
Decision Tree Precision: 78.26%
Decision Tree Recall: 81.82%
Decision Tree F1-Score: 80.00%

All 5 figures generated successfully!
